In [2]:
import warnings
import pandas as pd
import gtfs_kit as gk
from typing import List, Tuple, Set
import copy
from pathlib import Path
import re

In [3]:
gtfs_output_path = Path("/home/simpal/otp/data")

gtfs_year = 2025
gtfs_root = Path("/home/simpal/O/sharing-trans-data/GTFS Data/CLEAN - GTFS DATA/" + str(gtfs_year))

if not gtfs_root.is_dir():
    print("INPUT ERROR: Directory not found: " + str(gtfs_root))
# Recursively find zip files
gtfs_files = list(gtfs_root.rglob("*.zip"))

# Extract YYYYMMDD from filename
def extract_date(path):
    match = re.search(r"\d{8}", path.name)
    return pd.to_datetime(match.group(), format="%Y%m%d") if match else None


gtfs_release = (
    pd.DataFrame({
        "path": gtfs_files,
        "file": [p.name for p in gtfs_files],
        "date": [extract_date(p) for p in gtfs_files],
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
    .assign(
        date_end = lambda df: df["date"].shift(-1),
        date_days = lambda df: (df["date"] - pd.Timestamp("1970-01-01")).dt.days,
        date_end_days = lambda df: (df["date_end"] - pd.Timestamp("1970-01-01")).dt.days,
    )
)


print(f"Total number GTFS files: {len(gtfs_release)}")

Total number GTFS files: 24


In [4]:
gtfs_list = {}
#Read all gtfs_files
for _, row in gtfs_release.iterrows():
    print(f"Reading GTFS file: {row['file']}")

    feed = gk.feed.read_feed(row["path"], dist_units = "m") #Not sure what dist_unit is.

    gtfs_list[row["file"]] = feed


Reading GTFS file: GTFS_20250102.zip
Reading GTFS file: GTFS_20250113.zip
Reading GTFS file: GTFS_20250127.zip
Reading GTFS file: GTFS_20250210.zip
Reading GTFS file: GTFS_20250224.zip
Reading GTFS file: GTFS_20250310.zip
Reading GTFS file: GTFS_20250324.zip
Reading GTFS file: GTFS_20250407.zip
Reading GTFS file: GTFS_20250422.zip
Reading GTFS file: GTFS_20250505.zip
Reading GTFS file: GTFS_20250519.zip
Reading GTFS file: GTFS_20250616.zip
Reading GTFS file: GTFS_20250630.zip
Reading GTFS file: GTFS_20250728.zip
Reading GTFS file: GTFS_20250811.zip
Reading GTFS file: GTFS_20250825.zip
Reading GTFS file: GTFS_20250908.zip
Reading GTFS file: GTFS_20250922.zip
Reading GTFS file: GTFS_20251006.zip
Reading GTFS file: GTFS_20251020.zip
Reading GTFS file: GTFS_20251103.zip
Reading GTFS file: GTFS_20251117.zip
Reading GTFS file: GTFS_20251201.zip
Reading GTFS file: GTFS_20251215.zip


In [5]:
#Function to trunceate GTFS feed at specified date
#Cutoff  date will not be included
def truncate_feed_to_date(feed_i, cutoff_date):
    cutoff_str = cutoff_date.strftime("%Y%m%d")

    if feed_i.calendar is not None:
        cal = feed_i.calendar.copy()
        cal = cal[cal.start_date < cutoff_str]
        cal.loc[cal.end_date >= cutoff_str, "end_date"] = cutoff_str
        feed_i.calendar = cal
        del cal

    if feed_i.calendar_dates is not None:
        cd = feed_i.calendar_dates.copy()
        cd = cd[cd.date < cutoff_str]
        feed_i.calendar_dates = cd
        del cd

    #filter trips using valid service_id
    valid_service_ids = set()
    if feed_i.calendar is not None:
        valid_service_ids.update(feed_i.calendar.service_id.unique())

    if feed_i.calendar_dates is not None:
        valid_service_ids.update(feed_i.calendar_dates.service_id.unique())

    feed_i.trips = feed_i.trips[
        feed_i.trips.service_id.isin(valid_service_ids)
    ]

    #restrict_to_trips: Build a new feed by restricting this one to only the stops, trips, shapes, etc. used by the trips of the given IDs. Return the resulting feed.
    feed_trunc = gk.miscellany.restrict_to_trips(feed_i, feed_i.trips.trip_id.tolist())


    return feed_trunc

In [6]:
for i, row in gtfs_release.iterrows():
    file_key = row["file"]
    cutoff = row["date_end"]

    if pd.isna(cutoff):
        continue

    print(f"Truncating {file_key} to {cutoff.date()}")
    gtfs_list[file_key] = truncate_feed_to_date(gtfs_list[file_key], cutoff)


Truncating GTFS_20250102.zip to 2025-01-13
Truncating GTFS_20250113.zip to 2025-01-27
Truncating GTFS_20250127.zip to 2025-02-10
Truncating GTFS_20250210.zip to 2025-02-24
Truncating GTFS_20250224.zip to 2025-03-10
Truncating GTFS_20250310.zip to 2025-03-24
Truncating GTFS_20250324.zip to 2025-04-07
Truncating GTFS_20250407.zip to 2025-04-22
Truncating GTFS_20250422.zip to 2025-05-05
Truncating GTFS_20250505.zip to 2025-05-19
Truncating GTFS_20250519.zip to 2025-06-16
Truncating GTFS_20250616.zip to 2025-06-30
Truncating GTFS_20250630.zip to 2025-07-28
Truncating GTFS_20250728.zip to 2025-08-11
Truncating GTFS_20250811.zip to 2025-08-25
Truncating GTFS_20250825.zip to 2025-09-08
Truncating GTFS_20250908.zip to 2025-09-22
Truncating GTFS_20250922.zip to 2025-10-06
Truncating GTFS_20251006.zip to 2025-10-20
Truncating GTFS_20251020.zip to 2025-11-03
Truncating GTFS_20251103.zip to 2025-11-17
Truncating GTFS_20251117.zip to 2025-12-01
Truncating GTFS_20251201.zip to 2025-12-15


In [7]:
# Configuration: primary table name as key, with id column stored in id_col
ID_CONFIG = {
    "stops": {
        "id_col": "stop_id",
        "identity_cols": ["stop_lat", "stop_lon", "stop_name", "stop_id"], #stops only table with is primary key included in identity_cols. This is because stops are often moved somewhat, but I've been promised they don't change stop_id unless they move it more than 40 meters.
        "foreign_keys": [
            ("stop_times", "stop_id"),
            ("transfers", "from_stop_id"),
            ("transfers", "to_stop_id"),
        ],
    },

    "shapes": {
        "id_col": "shape_id",
        "identity_cols": ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
        "foreign_keys": [
            ("trips", "shape_id"),
        ],
    },

    "agency": {
        "id_col": "agency_id",
        "identity_cols": ["agency_name", "agency_timezone"],
        "foreign_keys": [
            ("routes", "agency_id"),
        ],
    },

    "routes": {
        "id_col": "route_id",
        "identity_cols": ["agency_id", "route_short_name", "route_type"],
        "foreign_keys": [
            ("trips", "route_id"),
            ("transfers", "from_route_id"),
            ("transfers", "to_route_id"),
        ],
    },

    "trips": {
        "id_col": "trip_id",
        "identity_cols": ["service_id", "trip_headsign", "trip_short_name", "direction_id"],
        "foreign_keys": [
            ("stop_times", "trip_id"),
            ("transfers", "from_trip_id"),
            ("transfers", "to_trip_id"),
        ],
    },
}

ID_CONFIG_service_id = {
    "calendar": {
        "id_col": "service_id",
        "identity_cols": [
            "monday", "tuesday", "wednesday", "thursday", "friday",
            "saturday", "sunday", "start_date", "end_date"
        ],
        "foreign_keys": [
            ("trips", "service_id"),
            ("calendar_dates", "service_id"),
        ],
    }
}

In [8]:
print("\nMerging all feeds into combined GTFS feed...")
if "combined_feed" in globals(): #in case of rerun
    del combined_feed

feed_names = list(gtfs_list.keys())
combined_feed = copy.deepcopy(gtfs_list[feed_names[0]])
print(f"Starting with base feed: {feed_names[0]}")

# Add feed_id to the inital feed tables (same way you do for subsequent feeds)
for table in [
    "agency", "routes", "stops", "trips", "stop_times",
    "calendar", "calendar_dates", "shapes", "transfers"
]:
    df = getattr(combined_feed, table, None)
    setattr(combined_feed, table, df.assign(feed_id=feed_names[0]))

for feed_name in feed_names[1:]:
    print(f"Merging feed: {feed_name}")
    feed_to_merge = gtfs_list[feed_name]

    combined_feed.agency = pd.concat([combined_feed.agency, feed_to_merge.agency.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.routes = pd.concat([combined_feed.routes, feed_to_merge.routes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stops = pd.concat([combined_feed.stops, feed_to_merge.stops.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.trips = pd.concat([combined_feed.trips, feed_to_merge.trips.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stop_times = pd.concat([combined_feed.stop_times, feed_to_merge.stop_times.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar = pd.concat([combined_feed.calendar, feed_to_merge.calendar.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar_dates = pd.concat([combined_feed.calendar_dates, feed_to_merge.calendar_dates.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.shapes = pd.concat([combined_feed.shapes, feed_to_merge.shapes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.transfers = pd.concat([combined_feed.transfers, feed_to_merge.transfers.assign(feed_id = feed_name)], ignore_index=True)

#    del gtfs_list[feed_name]
    del feed_to_merge


print(f"\nMerge complete! Combined feed statistics:")
if combined_feed.agency is not None:
    print(f"  Agencies: {len(combined_feed.agency)}")
if combined_feed.routes is not None:
    print(f"  Routes: {len(combined_feed.routes)}")
if combined_feed.stops is not None:
    print(f"  Stops: {len(combined_feed.stops)}")
if combined_feed.trips is not None:
    print(f"  Trips: {len(combined_feed.trips)}")
if combined_feed.stop_times is not None:
    print(f"  Stop times: {len(combined_feed.stop_times)}")
if combined_feed.calendar is not None:
    print(f"  Calendar entries: {len(combined_feed.calendar)}")
if combined_feed.shapes is not None:
    print(f"  Shape points: {len(combined_feed.shapes)}")


Merging all feeds into combined GTFS feed...
Starting with base feed: GTFS_20250102.zip
Merging feed: GTFS_20250113.zip
Merging feed: GTFS_20250127.zip
Merging feed: GTFS_20250210.zip
Merging feed: GTFS_20250224.zip
Merging feed: GTFS_20250310.zip
Merging feed: GTFS_20250324.zip
Merging feed: GTFS_20250407.zip
Merging feed: GTFS_20250422.zip
Merging feed: GTFS_20250505.zip
Merging feed: GTFS_20250519.zip
Merging feed: GTFS_20250616.zip
Merging feed: GTFS_20250630.zip
Merging feed: GTFS_20250728.zip
Merging feed: GTFS_20250811.zip
Merging feed: GTFS_20250825.zip
Merging feed: GTFS_20250908.zip
Merging feed: GTFS_20250922.zip
Merging feed: GTFS_20251006.zip
Merging feed: GTFS_20251020.zip
Merging feed: GTFS_20251103.zip
Merging feed: GTFS_20251117.zip
Merging feed: GTFS_20251201.zip
Merging feed: GTFS_20251215.zip

Merge complete! Combined feed statistics:
  Agencies: 479
  Routes: 38643
  Stops: 892866
  Trips: 4392418
  Stop times: 103083671
  Calendar entries: 37747
  Shape points: 8

In [9]:
len(gtfs_list[feed_names[0]].calendar)

1101

In [10]:
(combined_feed.calendar
    .drop(columns=['start_date', 'end_date', 'feed_id'])
    .groupby('service_id')
    .apply(lambda g: g.duplicated(keep=False).sum())
)

service_id
1       20
10      19
100     21
1000    21
1001    19
        ..
995     23
996     22
997     20
998     19
999     21
Length: 2050, dtype: int64

In [13]:
combined_feed_org = copy.deepcopy(combined_feed)

In [14]:
def find_conflicting_ids(feed, id_col, primary_table, identity_cols):
    df = getattr(feed, primary_table, None)
    if df is None:
        raise ValueError(f"Error: {primary_table} not found in feed")

    if any(col not in df.columns for col in identity_cols):
        raise ValueError(f"Error: one or more identity_cols not found in {primary_table}")

    use_identity_cols = identity_cols + [id_col]

    # Group by ID and count unique definitions
    id_definitions = (
        df
        .drop_duplicates(subset=use_identity_cols)
        .groupby(id_col)
        .size()
    )
    conflicting = set(id_definitions[id_definitions > 1].index)
    return conflicting


def apply_prefix_to_feed(feed, id_col, conflicting_ids, foreign_keys, primary_table):

    if not conflicting_ids:
        return

    # All tables/columns to update (primary + foreign keys)
    tables_to_update = [(primary_table, id_col)]
    tables_to_update.extend(foreign_keys)

    # Collect all conflicting IDs that exist in this feed
    feed_conflicting_ids = set()

    for table_name, col_name in tables_to_update:
        df = getattr(feed, table_name, None)
        if df is None or col_name not in df.columns:
            continue

        existing_ids = set(df[col_name].dropna().unique())
        feed_conflicting_ids.update(existing_ids & conflicting_ids)


    # Apply prefix to all tables
    for table_name, col_name in tables_to_update:
        df = getattr(feed, table_name, None)
        if df is None or col_name not in df.columns:
            continue

        mask = df[col_name].isin(feed_conflicting_ids)
        #add prefix from feed_id (feed_id was added to the combined_feed above)
        if mask.any():
            if "feed_id" not in df.columns:
                raise ValueError(f"{table_name} is missing feed_id")

            df.loc[mask, col_name] = (
                df.loc[mask, "feed_id"].astype(str)
                .str.replace("GTFS_", "", regex=False)
                .str.replace(".zip", "", regex=False)
                + "_"
                + df.loc[mask, col_name].astype(str)
            )

In [17]:
#add prefix to non-unique service_id. Rest ID are prefixed later, but since service_id is especially inconsistent across feed over time, prefix is added before trying to drop duplicates later.
primary_table, config = next(iter(ID_CONFIG_service_id.items()))
print(f"\nProcessing {primary_table}...")
conflicting = find_conflicting_ids(
    combined_feed,
    config["id_col"],
    primary_table,
    config["identity_cols"]
)
print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
if conflicting:
    apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], primary_table)

    print(f"    Prefixed conflicting {config["id_col"]} in all feeds")


Processing calendar...
  Found 1953 conflicting service_id values
    Prefixed conflicting service_id in all feeds


In [34]:
def deduplicate_feed(feed: gk.feed.Feed, id_col: str, primary_table: str, identity_cols: List[str],
                     foreign_keys: List[Tuple[str, str]]) -> int:
    df_primary = getattr(feed, primary_table, None)
    if df_primary is None or df_primary.empty:
        raise ValueError(f"Error: {primary_table} not found in feed")

    initial_count = len(df_primary)

    # Filter identity_cols to those present in the dataframe
    use_identity_cols = [c for c in identity_cols if c in df_primary.columns and c != id_col]

    if not use_identity_cols:
        raise ValueError(f"No identity columns found for {primary_table}")

    if primary_table in ["shapes", "stop_times"]:
        if primary_table == "shapes":
            sequence_cols = ["shape_pt_sequence"]
        if primary_table == "stop_times":
            sequence_cols = ["stop_sequence"]
        # For sequence tables: create signature from all rows grouped by ID
        sort_col = next((c for c in sequence_cols if c in df_primary.columns), None) #sequence column
        df_sorted = df_primary.sort_values([id_col, sort_col])
        
        # Create row-level signature by concatenating identity columns
        df_sorted['_row_sig'] = pd.util.hash_pandas_object(
            df_sorted[use_identity_cols],
            index = False
        )

        # Group and concatenate row signatures into single signature per ID
        signatures = (
            df_sorted
            .groupby(id_col)["_row_sig"]
            .apply(tuple)
            .map(hash)
            .rename("_signature")
            .reset_index()
        )

        # Map signature to canonical (minimum) ID
        canonical_map = (
            signatures
            .groupby('_signature')[id_col]
            .min()
        )

        # Create ID to canonical ID mapping
        id_to_canonical = (
            signatures
            .set_index(id_col)['_signature']
            .map(canonical_map)
        ) #For foreign key

        # Get set of canonical IDs
        canonical_ids = canonical_map.values

        # Update primary table: keep only rows with canonical IDs
        setattr(
            feed,
            primary_table,
            df_primary[df_primary[id_col].isin(canonical_ids)].reset_index(drop=True)
        )
    elif primary_table == "trips":
        df_st = feed.stop_times.sort_values(["trip_id", "stop_sequence"])

        df_st['_row_sig'] = pd.util.hash_pandas_object(
            df_st[["stop_id", "arrival_time", "departure_time"]],
            index = False
        )

        pattern = (
            df_st.groupby("trip_id")["_row_sig"]
            .apply(tuple)
            .map(hash)
            .rename("pattern_sig")
        )
        df_st = df_st.drop(columns=["_row_sig"]) # save memory
        df_primary["pattern_sig"] = df_primary["trip_id"].map(pattern)

        df_primary["trip_sig"] = pd.util.hash_pandas_object(
            df_primary[["route_id", "service_id", "direction_id", "shape_id", "pattern_sig"]],
            index = False
        )

        canonical = (
            df_primary
            .groupby("trip_sig")["trip_id"]
            .min()
        )

        trip_map = (
            df_primary.set_index("trip_id")["trip_sig"]
            .map(canonical)
        )

        df_primary["trip_id"] = df_primary["trip_id"].map(trip_map).fillna(df_primary["trip_id"])
        df_st["trip_id"] = df_st["trip_id"].map(trip_map).fillna(df_st["trip_id"])

        df_primary = df_primary.drop_duplicates("trip_id")

        df_st = (
            df_st
            .sort_values(["trip_id", "stop_sequence"])
            .drop_duplicates(["trip_id", "stop_sequence"])
        )
        df_primary = (
            df_primary
            .sort_values(["trip_id"])
            .drop_duplicates(["trip_id"])
            .drop(columns=["pattern_sig", "trip_sig"], errors="ignore")
        )

        df_primary = df_primary.drop(columns=["pattern_sig", "trip_sig"], errors="ignore")

        feed.trips = df_primary.reset_index(drop=True)
        feed.stop_times = df_st.reset_index(drop=True)

        final_count = len(df_primary)
        duplicates_removed = initial_count - final_count
        return duplicates_removed #don't run foreign key loop has it has been done manually for trips

    elif primary_table == "stops": #coordinate of stops sometimes changes a little bit. I've been told they keep stop_id consistent (!) and only change it when it is moved more than 40 m
        df_sorted = df_primary.sort_values(id_col)

        lat_threshold = 0.0003592535
        lon_threshold = 0.00064109755

        max_lat_delta = (
            df_sorted.groupby("stop_id")["stop_lat"]
            .transform(lambda x: (x.mean() - x).abs().max())
        )
        max_lon_delta = (
            df_sorted.groupby("stop_id")["stop_lon"]
            .transform(lambda x: (x.mean() - x).abs().max())
        )
        df_sorted["stable_loc"] = (max_lat_delta < lat_threshold) &  (max_lon_delta < lon_threshold)  #~40m (~56.6m in diagonal movement) at 56N

        stable_means = (
            df_sorted.loc[df_sorted["stable_loc"]]
            .groupby("stop_id", as_index=True)[["stop_lat", "stop_lon"]]
            .mean()
        )

        # write back means only for stable rows
        stable_mask = df_sorted["stable_loc"]
        df_sorted.loc[stable_mask, "stop_lat"] = df_sorted.loc[stable_mask, "stop_id"].map(stable_means["stop_lat"])
        df_sorted.loc[stable_mask, "stop_lon"] = df_sorted.loc[stable_mask, "stop_id"].map(stable_means["stop_lon"])

        # warning flags
        high_lat_delta = (max_lat_delta >= lat_threshold)
        high_lon_delta = (max_lon_delta >= lon_threshold)
        if high_lat_delta.any() or high_lon_delta.any():
            print(
                "Warning: Stops with max delta lat/lon above 40m threshold:\n",
                df_sorted.loc[high_lat_delta | high_lon_delta, ["stop_id", "stop_lat", "stop_lon"]].head(10),
                "\nDuplicated stop_id with lat/lon differences > 40, will get new stop_id."
            )

        #drop duplicates with same stop_id/_lat/_lon. lat/lon has been average by stop_id if within 40m. Duplicated stop_id with delta lat/lon, will get new stop_id below.
        df_sorted = df_sorted.drop_duplicates(subset=["stop_id", "stop_lat", "stop_lon"], inplace=False)


        feed.stops = df_sorted.reset_index(drop=True)
        final_count = len(df_sorted)
        duplicates_removed = initial_count - final_count
        return duplicates_removed #don't run foreign key loop has it has been done manually for trips
    else:
        # For simple tables: group by identity columns directly
        df_sorted = df_primary.sort_values(id_col)

        # Map each unique combination of identity cols to canonical (minimum) ID
        canonical_df = (
            df_sorted
            .groupby(use_identity_cols, dropna=False)[id_col]
            .min()
            .reset_index()
        )

        # Create mapping from all IDs to canonical IDs
        id_to_canonical = (
            df_sorted
            .merge(canonical_df, on=use_identity_cols, suffixes=('', '_canonical'))
            .set_index(id_col)[f'{id_col}_canonical']
        ) #For foreign key

        # Update primary table: keep only canonical rows
        canonical_ids = canonical_df[id_col]
        setattr(feed, primary_table, df_primary[df_primary[id_col].isin(canonical_ids)].reset_index(drop=True))

    # Update all foreign key references
    for fk_table, fk_col in foreign_keys:
        fk_df = getattr(feed, fk_table, None)
        if fk_df is None or fk_col not in fk_df.columns:
            warnings.warn(f"Warning: Foreign key column {fk_col} not found in {fk_table}. Skipping foreign key update.")
            continue

        # Map foreign keys to canonical IDs
        fk_df[fk_col] = fk_df[fk_col].map(id_to_canonical).fillna(fk_df[fk_col])
        setattr(feed, fk_table, fk_df)

    final_count = len(getattr(feed, primary_table))
    duplicates_removed = initial_count - final_count

    return duplicates_removed


print("Starting deduplication process...")
print(f"\nBefore deduplication:")
for table in ['stops', 'stop_times', 'shapes', 'routes', 'agency']:
    if getattr(combined_feed, table, None) is not None:
        print(f"  {table}: {len(getattr(combined_feed, table, None))}")

# Apply deduplication for each ID type
for primary_table, config in ID_CONFIG.items(): #Important calendar and calendar_dates are not included in removing duplicates!
    print(f"\nDeduplicating {primary_table}...")

    removed = deduplicate_feed(
        combined_feed,
        config["id_col"],
        primary_table,
        config["identity_cols"],
        config["foreign_keys"]
    )

    df = getattr(combined_feed, primary_table, None)
    if df is not None:
        print(f"  {primary_table}: removed {removed} duplicates, {len(df)} remaining")

print("\n" + "=" * 50)
print("Deduplication complete!")
print(f"\nAfter deduplication:")
for table in ['stops', 'stop_times', 'shapes', 'routes', 'agency']:
    df = getattr(combined_feed, table, None)
    if df is not None:
        print(f"  {table}: {len(df)}")

del df

Starting deduplication process...

Before deduplication:
  stops: 892866
  stop_times: 103083671
  shapes: 89428529
  routes: 38643
  agency: 479

Deduplicating stops...
              stop_id   stop_lat   stop_lon
377035  000000000006  55.755782  12.494917
637876  000000000006  55.755782  12.494917
749589  000000000006  55.755782  12.494917
675107  000000000006  55.755782  12.494917
451983  000000000006  55.755782  12.494917
339643  000000000006  55.755782  12.494917
302307  000000000006  55.755782  12.494917
79202   000000000006  55.755782  12.494917
526409  000000000006  55.755782  12.494917
264961  000000000006  55.755782  12.494917 
Duplicated stop_id with lat/lon differences > 40, will get new stop_id.
  stops: removed 853327 duplicates, 39539 remaining

Deduplicating shapes...


KeyboardInterrupt: 

In [ ]:
def merge_calendar(feed): #Not tested
    """
    Merge calendar rows with same service_id and weekday pattern
    if their date periods are consecutive or overlapping.
    Keeps earliest start_date and latest end_date.
    """
    if feed.calendar is None or feed.calendar.empty:
        return

    cal = feed.calendar.copy()

    # Columns that define the "service pattern" (excluding dates)
    weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    group_cols = ["service_id"] + [c for c in weekday_cols if c in cal.columns]

    # Convert dates to datetime for comparison
    cal["_start"] = pd.to_datetime(cal["start_date"], format="%Y%m%d")
    cal["_end"] = pd.to_datetime(cal["end_date"], format="%Y%m%d")

    # Sort by service pattern and start date
    cal = cal.sort_values(group_cols + ["_start"]).reset_index(drop=True)

    merged_rows = []

    for _, group in cal.groupby(group_cols, dropna=False):
        group = group.sort_values("_start").reset_index(drop=True)

        # Start with first row
        current_start = group.loc[0, "_start"]
        current_end = group.loc[0, "_end"]
        current_row = group.loc[0].copy()

        for i in range(1, len(group)):
            row_start = group.loc[i, "_start"]
            row_end = group.loc[i, "_end"]

            # Check if consecutive or overlapping (1 day gap = consecutive)
            if row_start <= current_end + pd.Timedelta(days=1):
                # Extend the period
                current_end = max(current_end, row_end)
            else:
                # Gap too large — save current and start new period
                current_row["start_date"] = current_start.strftime("%Y%m%d")
                current_row["end_date"] = current_end.strftime("%Y%m%d")
                merged_rows.append(current_row)

                current_start = row_start
                current_end = row_end
                current_row = group.loc[i].copy()

        # Save final period
        current_row["start_date"] = current_start.strftime("%Y%m%d")
        current_row["end_date"] = current_end.strftime("%Y%m%d")
        merged_rows.append(current_row)

    # Build merged dataframe
    merged_cal = pd.DataFrame(merged_rows)
    merged_cal = merged_cal.drop(columns=["_start", "_end"], errors="ignore")

    # Preserve original column order
    merged_cal = merged_cal[feed.calendar.columns]

    initial_count = len(feed.calendar)
    feed.calendar = merged_cal.reset_index(drop=True)
    final_count = len(feed.calendar)

    print(f"  calendar: merged {initial_count} → {final_count} rows")

In [ ]:
# Apply to prefix to all ID types
for primary_table, config in ID_CONFIG.items():
    print(f"\nProcessing {primary_table}...")
    conflicting = find_conflicting_ids(
        combined_feed,
        config["id_col"],
        primary_table,
        config["identity_cols"]
    )
    print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
    if conflicting:
        apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], config["primary_table"])
        print(f"    Prefixed conflicting {config["id_col"]} in all feeds")